# 04. Implementación de la Query Tower (Torre de Consulta)

Este cuaderno implementa la **Fase 2** del plan de desarrollo para el recomendador Two Towers.
El objetivo es construir y validar la **Query Tower**, el componente encargado de procesar la información del cliente y su contexto de geolocalización y operativa para proyectarlos en un espacio vectorial denso de dimensión $D = 128$.

### Objetivos:
1. Cargar el dataset de entrenamiento y validación.
2. Extraer los vocabularios únicos para las capas `StringLookup`.
3. Configurar el pipeline de datos (`tf.data.Dataset`).
4. Implementar la clase `QueryTower` heredando de `tf.keras.Model`.
5. Configurar la normalización de variables de geolocalización (`LATITUD` y `LONGITUD`).
6. Verificar el paso forward del modelo y las dimensiones del vector resultante.


In [ ]:
import polars as pl
import tensorflow as tf
import tensorflow_recommenders as tfrs
import numpy as np
import os

print("TensorFlow versión:", tf.__version__)
print("TensorFlow Recommenders versión:", tfrs.__version__)


## 1. Carga de Datasets
Cargamos el set de entrenamiento generado en la Fase 1 y el catálogo de productos para extraer la información necesaria para los vocabularios y la normalización de geoubicación.


In [ ]:
# Cargar datos procesados con Polars
train_path = "data_processed/retrieval_train.parquet"
val_path = "data_processed/retrieval_val.parquet"
prd_path = "data_processed/products_catalog.parquet"

train_df = pl.read_parquet(train_path)
val_df = pl.read_parquet(val_path)
products_df = pl.read_parquet(prd_path)

print(f"Dataset de Entrenamiento: {train_df.height:,} pares")
print(f"Dataset de Validación: {val_df.height:,} pares")
print(f"Catálogo de Productos: {products_df.height:,} productos")


## 2. Extracción de Vocabularios Únicos
Para inicializar las capas `StringLookup` de Keras de manera eficiente, extraemos los vocabularios únicos en Polars y los convertimos en listas de Python. Esto optimiza el consumo de memoria y el tiempo de arranque.


In [ ]:
# Vocabulario de RUC (Clientes)
vocab_ruc = train_df["RUC"].unique().to_list()
# Vocabulario de CIUDAD
vocab_ciudad = train_df["CIUDAD"].unique().to_list()
# Vocabulario de RUTA
vocab_ruta = train_df["RUTA"].unique().to_list()
# Vocabulario de COD_PROD (Semilla) - Usamos el catálogo completo para cold-start
vocab_products = products_df["id_producto"].unique().to_list()

print("Estadísticas de Vocabularios:")
print(f" - Clientes únicos (RUC): {len(vocab_ruc):,}")
print(f" - Ciudades únicas: {len(vocab_ciudad)}")
print(f" - Rutas únicas: {len(vocab_ruta)}")
print(f" - Productos únicos en Catálogo: {len(vocab_products):,}")


## 3. Pipeline de Datos en TensorFlow (tf.data.Dataset)
Convertimos los DataFrames de Polars a tensores de TensorFlow y estructuramos un pipeline eficiente usando `tf.data.Dataset`.
Definiremos un diccionario de inputs para el Query que contiene:
- `RUC`
- `CIUDAD`
- `RUTA`
- `LATITUD`
- `LONGITUD`
- `COD_PROD` (Producto Semilla / Trigger)

El target positivo de co-compra en esta fase de retrieval es el producto complementario `COD_PROD_2`.


In [ ]:
def make_tf_dataset(df: pl.DataFrame, batch_size: int = 1024, shuffle: bool = False) -> tf.data.Dataset:
    # Agrupamos las características del Query
    inputs = {
        "RUC": df["RUC"].to_numpy(),
        "CIUDAD": df["CIUDAD"].to_numpy(),
        "RUTA": df["RUTA"].to_numpy(),
        "LATITUD": df["LATITUD"].to_numpy().astype(np.float32),
        "LONGITUD": df["LONGITUD"].to_numpy().astype(np.float32),
        "COD_PROD": df["COD_PROD"].to_numpy(),
    }
    
    # El label de entrenamiento es el producto de co-compra (positivo)
    targets = df["COD_PROD_2"].to_numpy()
    
    ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
    
    if shuffle:
        ds = ds.shuffle(buffer_size=100_000)
    
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Crear un lote de ejemplo para pruebas de inferencia
train_ds = make_tf_dataset(train_df, batch_size=512, shuffle=False)
example_batch = next(iter(train_ds))

print("Estructura del lote de ejemplo:")
print("Inputs keys:", list(example_batch[0].keys()))
print("Inputs RUC shape:", example_batch[0]["RUC"].shape)
print("Targets shape:", example_batch[1].shape)


## 4. Arquitectura de la Query Tower (Keras Model)
Definimos la Query Tower heredando de `tf.keras.Model`.
Esta torre:
1. Codifica cada característica categórica (`RUC`, `CIUDAD`, `RUTA`, `COD_PROD`) con `StringLookup` y `Embedding`.
2. Concatena latitud y longitud, y las normaliza usando `tf.keras.layers.Normalization`.
3. Concatena todos los embeddings y las variables continuas normalizadas.
4. Pasa la concatenación por una MLP profunda (red densa con ReLU y Dropout).
5. Proyecta el resultado final a un vector de dimensión $D$ mediante una capa lineal.


In [ ]:
class QueryTower(tf.keras.Model):
    def __init__(
        self,
        vocab_ruc,
        vocab_ciudad,
        vocab_ruta,
        vocab_products,
        embedding_dim: int = 128,
        dropout_rate: float = 0.2
    ):
        super().__init__()
        
        # 1. Embeddings para Categóricas (StringLookup + Embedding)
        # Cliente (RUC)
        self.ruc_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruc, mask_token=None)
        self.ruc_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_ruc) + 1,
            output_dim=64,
            name="ruc_emb"
        )
        
        # Ciudad
        self.ciudad_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ciudad, mask_token=None)
        self.ciudad_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_ciudad) + 1,
            output_dim=16,
            name="ciudad_emb"
        )
        
        # Ruta
        self.ruta_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruta, mask_token=None)
        self.ruta_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_ruta) + 1,
            output_dim=32,
            name="ruta_emb"
        )
        
        # Producto Semilla (COD_PROD)
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_products) + 1,
            output_dim=64,
            name="product_emb"
        )
        
        # 2. Normalización de Geoubicación
        self.geo_normalization = tf.keras.layers.Normalization(axis=-1)
        
        # 3. Capas MLP de Interacción Profunda
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="query_projection")
        ])
        
    def call(self, inputs):
        # Procesar categóricas
        ruc_emb = self.ruc_embedding(self.ruc_lookup(inputs["RUC"]))
        ciudad_emb = self.ciudad_embedding(self.ciudad_lookup(inputs["CIUDAD"]))
        ruta_emb = self.ruta_embedding(self.ruta_lookup(inputs["RUTA"]))
        product_emb = self.product_embedding(self.product_lookup(inputs["COD_PROD"]))
        
        # Procesar continuas: Concatenar latitud y longitud y normalizar
        lat = tf.expand_dims(inputs["LATITUD"], axis=-1)
        lon = tf.expand_dims(inputs["LONGITUD"], axis=-1)
        geo_features = tf.concat([lat, lon], axis=-1)
        geo_norm = self.geo_normalization(geo_features)
        
        # Concatenar todos los embeddings y características de entrada
        concatenated = tf.concat([
            ruc_emb,
            ciudad_emb,
            ruta_emb,
            product_emb,
            geo_norm
        ], axis=-1)
        
        # Pasar por la MLP
        return self.mlp(concatenated)


## 5. Inicialización del Modelo y Adaptación de Normalización
Instanciamos el modelo de Query Tower y calculamos la media y varianza de la ubicación (`LATITUD` y `LONGITUD`) sobre el conjunto de entrenamiento para alimentar la capa de normalización de geoubicación.


In [ ]:
# Instanciar la Query Tower con dimensión de embedding final D = 128
query_tower = QueryTower(
    vocab_ruc=vocab_ruc,
    vocab_ciudad=vocab_ciudad,
    vocab_ruta=vocab_ruta,
    vocab_products=vocab_products,
    embedding_dim=128,
    dropout_rate=0.2
)

# Adaptar la capa de normalización de geoubicación con datos de entrenamiento
geo_train = train_df.select(["LATITUD", "LONGITUD"]).to_numpy().astype(np.float32)
query_tower.geo_normalization.adapt(geo_train)

print("Capa de normalización de geoubicación adaptada con éxito.")
print("Media calculada (Latitud, Longitud):", query_tower.geo_normalization.mean.numpy())
print("Varianza calculada (Latitud, Longitud):", query_tower.geo_normalization.variance.numpy())


## 6. Validación de Inferencia (Forward Pass)
Hacemos pasar nuestro lote de ejemplo por el modelo instanciado para asegurar que compila correctamente, no produce valores nulos/NaNs y devuelve la dimensión exacta requerida: `(batch_size, D)`.


In [ ]:
# Obtener inputs del batch de ejemplo
inputs_batch, _ = example_batch

# Ejecutar forward pass
query_embeddings = query_tower(inputs_batch)

print("Verificación de Dimensiones:")
for k, v in inputs_batch.items():
    print(f" - Input '{k}': {v.shape}")
print(f"\n- Salida de la Query Tower: {query_embeddings.shape}")

# Comprobar estabilidad numérica
num_nans = tf.reduce_sum(tf.cast(tf.math.is_nan(query_embeddings), tf.int32)).numpy()
print(f"- Cantidad de valores NaN en la salida: {num_nans}")

# Validar assertions
assert query_embeddings.shape == (512, 128), "Dimensiones incorrectas en la proyección del embedding de consulta."
assert num_nans == 0, "Se han generado valores numéricos inestables (NaN)."

print("\n¡Validación completada con éxito! La Query Tower funciona según las especificaciones de diseño.")


## 7. Resumen de Parámetros del Modelo
Visualizamos la estructura de parámetros y capas del modelo.


In [ ]:
# Llamada dummy para inicializar pesos y mostrar resumen
_ = query_tower(inputs_batch)
query_tower.summary()
